In [2]:
%use kandy

We are trying to estimate the performance ratio of stdlib functions in Kotlin/Native vs. JVM. For this example, we focus
on `String.substring` method used in this particular scenario. We need to exclude the effect of GC to get the performace
ratio of execution time of the function itself. The following cell describes, how we collect the data. We
end up with a list of execution times.

In [3]:
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardOpenOption

val kotlincNative = System.getProperty("user.home") + "/.konan/kotlin-native-prebuilt-linux-x86_64-2.2.10/bin/kotlinc-native"
val tempDir = Files.createTempDirectory("knative-perf")

fun runProcess(vararg command: String): String =
    ProcessBuilder(command.toList())
        .directory(tempDir.toFile())
        .start()
        .let { process ->
            process.inputStream.bufferedReader().readText()
                .also {
                    if (process.waitFor() != 0) {
                        process.errorStream.bufferedReader().readText().let(::println)
                        throw Exception("Process failed with code ${process.exitValue()}")
                    }
                }
        }

val source = """
import kotlin.concurrent.Volatile
import kotlin.time.measureTime
import kotlin.time.DurationUnit
import kotlin.time.Duration

@Volatile
private var blackhole: Any? = null
@Volatile
private var string = "abc".repeat(100000)

fun main() {
    repeat(1_000_000) { blackhole = string.substring(1000, 5000) }
    val iterations = 10_000
    val times = ArrayList<Double>(iterations)
    repeat(iterations) {
        measureTime {
            repeat(100) {
                blackhole = string.substring(1000, 5000)
            }
        }.let { times.add(it.toDouble(DurationUnit.SECONDS)) }
    }
    times.forEach { println(it) }
}
"""

val sourceFile = tempDir.resolve("benchmark.kt")
val outputBinary = tempDir.resolve("benchmark.kexe")

Files.writeString(
    sourceFile,
    source,
    StandardOpenOption.CREATE,
)

runProcess(
        kotlincNative,
        sourceFile.toString(),
        "-o", outputBinary.toString(),
)

val times = runProcess(outputBinary.toString())
    .split("\n")
    .mapNotNull { it.takeIf { it.isNotEmpty() }?.toDouble() }
    .toList()

For the exclusion of GC, we assume that the execution time distribution is bimodal (i .e. there are two peaks). The
first peak is formed by the executions that didn't trigger GC. The second peak is formed by the executions that
triggered GC. By finding a threshold for the execution time that lies between the two peaks, we can separate the
executions into those that ran GC and those which didn't.

In [4]:
val threshold = 0.00005

In [5]:
plot {
    histogram(times, binsOption = BinsOption.byNumber(100))
    vLine {
        xIntercept.constant(threshold)
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="2YbBus"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("2YbBus");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"x",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"count":[50.0,2406.0,4538.0,392.0,49.0,13.0,57.0,59.0,42.0,41.0,49.0,122.0,270.0,241.0,106.0,112.0,115.0,143.0,103.0,94.0,91.0,86.0,94.0,113.0,121.0,122.0,64.0,48.0,38.0,18.0,11.0,10.0,11.0,12.0,14.0,24.0,16.0,14.0,14.0,10.0,18.0,14.0,12.0,7.0,2.0,0.0,0.0,3.0,2.0,2.0,2.0,0.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0],
"x":[1.471358E-5,1.891746E-5,2.3121339999999998E-5,2.732522E-5,3.1529099999999996E-5,3.5732980000000004E-5,3.993686E-5,4.414074E-5,4.834462E-5,5.25485E-5,5.675238E-5,6.0956259999999995E-5,6.516013999999999E-5,6.936402E-5,7.35679E-5,7.777178E-5,8.197565999999999E-5,8.617953999999999E-5,9.038342E-5,9.45873E-5,9.879118E-5,1.0299505999999999E-4,1.0719894E-4,1.1140281999999999E-4,1.156067E-4,1.1981057999999999E-4,1.2401446E-4,1.2821834E-4,1.3242222E-4,1.3662610000000002E-4,1.4082998000000002E-4,1.4503386E-4,1.4923774E-4,1.5344162E-4,1.576455E-4,1.6184938000000001E-4,1.6605326E-4,1.7025714E-4,1.7446102000000002E-4,1.7866490000000002E-4,1.8286878E-4,1.8707266E-4,1.9127654E-4,1.9548042E-4,1.9968430000000002E-4,2.0388818E-4,2.0809206E-4,2.1229594E-4,2.1649982000000002E-4,2.207037E-4,2.2490758E-4,2.2911146E-4,2.3331534E-4,2.3751922000000002E-4,2.417231E-4,2.4592698E-4,2.5013085999999997E-4,2.5433474E-4,2.5853862E-4,2.627425E-4,2.6694638E-4,2.7115026E-4,2.7535414E-4,2.7955802E-4,2.837619E-4,2.8796577999999997E-4,2.9216965999999997E-4,2.9637353999999996E-4,3.0057741999999995E-4,3.047813E-4,3.0898518E-4,3.1318906E-4,3.1739294E-4,3.2159682E-4,3.2580069999999997E-4,3.3000457999999997E-4,3.3420845999999996E-4,3.3841234E-4,3.4261622E-4,3.468201E-4,3.5102398E-4,3.5522786E-4,3.5943174E-4,3.6363561999999997E-4,3.6783949999999997E-4,3.7204337999999996E-4,3.7624725999999995E-4,3.8045114E-4,3.8465502E-4,3.888589E-4,3.9306278E-4,3.9726666E-4,4.0147054E-4,4.0567441999999997E-4,4.0987829999999996E-4,4.1408217999999996E-4,4.1828606E-4,4.2248994E-4,4.2669382E-4,4.308977E-4]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
}]
}
},{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"xintercept":5.0E-5,
"geom":"vline",
"data":{
}
}],
"spec_id":"2"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizi

We then estimate the total time spent in GC by the following formula.

In [6]:
val (noGC, GC) = times.partition { it < threshold }
val avgNoGC = noGC.average()
GC.sumOf { it - avgNoGC } / times.sum()

0.4388735816832946

For measurements that run at least 100 000 iterations and don't print the times throught the execution (but all at once at the end), we estimate that GC is responsible for over 80% of the execution. That is insane. Therefore, we question the results.

* Can GC in K/N be responsible for such a high percentage of execution time? After all, this is very memory heavy benchmark.
* Is there an error in the benchmark setup, collection of the data or the processing?
* Is any one of our assumptions incorrect?
* Is this method for estimating GC time sound?